[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/05_normalization.ipynb)

# 05. Normalization

정규화 축과 통계량이 달라질 때 tensor가 어떻게 변하고 어떤 reduction이 생기는지 비교한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. BatchNorm2d

batch/channel 축 통계를 사용한다.


In [ ]:
x = torch.tensor(
    [[[[1., 2.], [3., 4.]], [[2., 4.], [6., 8.]]],
     [[[2., 3.], [4., 5.]], [[1., 3.], [5., 7.]]]],
    device=device,
)
bn = nn.BatchNorm2d(2, affine=False, track_running_stats=False).to(device)
y = bn(x)
print(y)


In [ ]:
_ = profile_call("BatchNorm2d", bn, x)


## 2. InstanceNorm → GroupNorm

통계를 sample별 또는 channel group별로 제한한다.


In [ ]:
inn = nn.InstanceNorm2d(2, affine=False, track_running_stats=False).to(device)
gn = nn.GroupNorm(1, 2, affine=False).to(device)

print("InstanceNorm:\n", inn(x))
print("GroupNorm:\n", gn(x))


In [ ]:
_ = profile_call("InstanceNorm", inn, x)
_ = profile_call("GroupNorm", gn, x)


## 3. LayerNorm

마지막 feature 축을 정규화한다.


In [ ]:
z = torch.tensor([[1., 2., 3., 4.], [2., 4., 6., 8.]], device=device)
ln = nn.LayerNorm(4, elementwise_affine=False).to(device)
print(ln(z))


In [ ]:
_ = profile_call("LayerNorm", ln, z)


## 4. RMSNorm

평균을 빼지 않고 RMS로 scale만 맞춘다.


In [ ]:
rms = nn.RMSNorm(4, elementwise_affine=False).to(device)
print(rms(z))

rms_explicit = z / torch.sqrt(z.pow(2).mean(dim=-1, keepdim=True) + 1e-5)
print("explicit close:", torch.allclose(rms(z), rms_explicit, atol=1e-4))


In [ ]:
_ = profile_call("RMSNorm", rms, z)


## 5. QK-Norm

attention의 Q/K를 head dimension에서 따로 normalize한다.


In [ ]:
q = torch.randn(1, 2, 4, 4, device=device)
k = torch.randn_like(q)

q_norm = F.rms_norm(q, (q.size(-1),))
k_norm = F.rms_norm(k, (k.size(-1),))

print("q RMS before:", q.pow(2).mean(-1).sqrt())
print("q RMS after :", q_norm.pow(2).mean(-1).sqrt())


In [ ]:
_ = profile_call("QK RMSNorm", lambda p, q_: (F.rms_norm(p, (p.size(-1),)), F.rms_norm(q_, (q_.size(-1),))), q, k)


## References and provenance

**[5.1] BatchNorm**
- 출처: Ioffe & Szegedy, Batch Normalization
- 이 노트북에서 가져온 부분: batch/channel statistics

**[5.2] GroupNorm**
- 출처: Wu & He, Group Normalization
- 이 노트북에서 가져온 부분: channel grouping

**[5.3] LayerNorm**
- 출처: Ba et al., Layer Normalization
- 이 노트북에서 가져온 부분: feature-axis normalization

**[5.4] RMSNorm**
- 출처: Zhang & Sennrich, Root Mean Square Layer Normalization
- 이 노트북에서 가져온 부분: RMS-only normalization

**[5.5] QK-Norm**
- 출처: modern ViT/DiT/LLM implementations including Krea-family reports
- 이 노트북에서 가져온 부분: Q/K normalization before attention
